# 🍇 Grape Berry Detection — ONNX Export
Entrena YOLOv8n en GPU de Google (~10 min) y exporta el `.onnx` listo para el browser.

**Pasos:**
1. Menú → Runtime → Change runtime type → **T4 GPU**
2. Pon tu Roboflow API Key en la celda siguiente
3. Runtime → Run all
4. Descarga el archivo `grape-berry.onnx` al final

In [ ]:
# ── 1. Instalar dependencias ──────────────────────────────────────────────────
!pip install ultralytics roboflow --quiet

In [ ]:
# ── 2. Configurar API Key de Roboflow ─────────────────────────────────────────
# Obtén la tuya en https://app.roboflow.com → perfil → API Key
ROBOFLOW_API_KEY = "PON_TU_KEY_AQUI"  # ← reemplaza esto

In [ ]:
# ── 3. Descargar dataset de uvas desde Roboflow Universe ─────────────────────
from roboflow import Roboflow

rf      = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("grape-detection").project("grape-detection-xcemr")
version = project.version(1)
dataset = version.download("yolov8")

print("Dataset location:", dataset.location)
print("Classes:", version.classes)

In [ ]:
# ── 4. Entrenar YOLOv8n (10 epochs en GPU T4 ≈ 8-12 minutos) ────────────────
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data    = f"{dataset.location}/data.yaml",
    epochs  = 10,
    imgsz   = 640,
    batch   = 16,
    device  = 0,       # GPU
    name    = "grape-berry",
    project = "/content/runs",
    exist_ok = True,
)

print("\n✅ Entrenamiento terminado")

In [ ]:
# ── 5. Exportar a ONNX ────────────────────────────────────────────────────────
from pathlib import Path
from ultralytics import YOLO

best_pt = "/content/runs/grape-berry/weights/best.pt"
export_model = YOLO(best_pt)

onnx_path = export_model.export(
    format   = "onnx",
    opset    = 17,
    imgsz    = 640,
    dynamic  = False,
    simplify = True,
)

# Renombrar para facilitar la descarga
import shutil
shutil.copy2(onnx_path, "/content/grape-berry.onnx")

print(f"\n✅ Exportado: /content/grape-berry.onnx")

# Mostrar clases del modelo
print(f"\nClases ({len(export_model.names)}):")
for i, name in export_model.names.items():
    print(f"  {i}: {name}")

In [ ]:
# ── 6. Descargar el archivo ───────────────────────────────────────────────────
from google.colab import files
files.download("/content/grape-berry.onnx")
print("\n👆 El archivo se descargó. Guárdalo en: public/models/grape-berry.onnx")